# Week 2: Embedding Pipeline & Classical Baseline
**Project**: Quantum Kernels on CLIP / BERT Embeddings

**What this notebook does:**
1. Loads SST-2 sentiment dataset from HuggingFace
2. Extracts BERT embeddings (768-dimensional [CLS] vectors)
3. Reduces to 8 dimensions via PCA for quantum encoding
4. Trains 3 classical SVM baselines (linear, RBF, polynomial)
5. Saves embeddings to disk for Week 3 (quantum kernel experiments)

**Runtime**: ~10 minutes total (BERT extraction is the slow step)  
**Cost**: Free on Google Colab T4

---
## Step 1: Install & import

In [ ]:
!pip install datasets transformers torch scikit-learn numpy --quiet

import numpy as np
import time, os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("All imports OK")
print(f"PyTorch: {torch.__version__} | GPU: {torch.cuda.is_available()}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

---
## Step 2: Load SST-2 dataset

SST-2 (Stanford Sentiment Treebank) is the standard binary sentiment benchmark.  
Label 0 = negative, Label 1 = positive.  
We take 1000 samples — enough to show results without long compute times.

In [ ]:
# Load SST-2 from HuggingFace datasets
print("Loading SST-2 dataset...")
dataset = load_dataset("glue", "sst2")

# Use 600 train + 200 validation samples for speed
# (use more for final paper results)
N_TRAIN_EXTRACT = 600
N_VAL_EXTRACT   = 200

train_data = dataset['train'].shuffle(seed=42).select(range(N_TRAIN_EXTRACT))
val_data   = dataset['validation'].shuffle(seed=42).select(range(N_VAL_EXTRACT))

print(f"Train split: {len(train_data)} samples")
print(f"  Negative: {sum(1 for x in train_data if x['label'] == 0)}")
print(f"  Positive: {sum(1 for x in train_data if x['label'] == 1)}")
print(f"\nValidation split: {len(val_data)} samples")
print(f"\nExample sentences:")
for i in range(3):
    label_name = 'positive' if train_data[i]['label'] == 1 else 'negative'
    print(f"  [{label_name}] {train_data[i]['sentence'][:80]}...")

---
## Step 3: Extract BERT embeddings

We use `bert-base-uncased` to extract the **[CLS] token embedding** for each sentence.  
The [CLS] token is a special token that aggregates sentence-level meaning.  
Output: 768-dimensional float vector per sentence.

**Why BERT and not a fine-tuned model?**  
We use a *frozen, pretrained* model intentionally. The question we're asking is:  
"Do quantum kernels add value on top of general-purpose embeddings?" — so we keep  
the encoder fixed and only vary the kernel.

In [ ]:
MODEL_NAME = "bert-base-uncased"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()  # disable dropout

print(f"Model loaded. Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

def extract_bert_embeddings(sentences, batch_size=32, show_progress=True):
    """Extract [CLS] token embeddings from BERT for a list of sentences."""
    all_embeddings = []
    n_batches = (len(sentences) + batch_size - 1) // batch_size

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            outputs = bert_model(**inputs)

        # [CLS] token = first token of last hidden state
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())

        if show_progress and (i // batch_size + 1) % 5 == 0:
            print(f"  Batch {i // batch_size + 1}/{n_batches} done...")

    return np.vstack(all_embeddings)

# Extract embeddings
print("\nExtracting train embeddings...")
t0 = time.time()
train_sentences = [x['sentence'] for x in train_data]
train_labels    = np.array([x['label'] for x in train_data])
train_emb = extract_bert_embeddings(train_sentences)
print(f"  Done in {time.time()-t0:.1f}s | Shape: {train_emb.shape}")

print("\nExtracting validation embeddings...")
t0 = time.time()
val_sentences = [x['sentence'] for x in val_data]
val_labels    = np.array([x['label'] for x in val_data])
val_emb = extract_bert_embeddings(val_sentences)
print(f"  Done in {time.time()-t0:.1f}s | Shape: {val_emb.shape}")

print(f"\nEmbedding stats:")
print(f"  Mean L2 norm: {np.mean(np.linalg.norm(train_emb, axis=1)):.3f}")
print(f"  Std  L2 norm: {np.std(np.linalg.norm(train_emb, axis=1)):.3f}")

---
## Step 4: PCA reduction

Quantum angle encoding requires the feature dimension to equal the qubit count.  
We reduce 768 → 8 dimensions. We also try 4, 12, 16 to study the tradeoff later.

**Why PCA and not random projection?**  
PCA preserves the maximum variance in the fewest dimensions. This means the reduced  
embeddings carry as much semantic information as possible into the quantum circuit.  
Random projection is faster but discards more structure.

**Important**: fit PCA on training data only, then transform both sets.

In [ ]:
# Fit PCA on training embeddings only
N_QUBITS = 8  # primary dimension for Week 3 experiments
pca_dims_to_try = [4, 8, 12, 16]

print("PCA variance explained by dimension:")
print(f"  {'Dims':>6}  {'Var Explained':>15}  {'Cumulative PCs for 80%':>22}")
print(f"  {'-'*50}")

pca_objects = {}
for n in pca_dims_to_try:
    pca = PCA(n_components=n, random_state=42)
    pca.fit(train_emb)
    var = np.sum(pca.explained_variance_ratio_) * 100
    pca_objects[n] = pca
    # How many PCs needed to explain 80%?
    pca_full = PCA(random_state=42).fit(train_emb)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_for_80 = np.searchsorted(cumvar, 0.8) + 1
    print(f"  {n:>6}  {var:>14.1f}%  {n_for_80:>22} (first time only)")

# Use 8-dim PCA as primary
pca_8 = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_val_pca   = pca_8.transform(val_emb)

print(f"\nPrimary reduction: 768 -> {N_QUBITS} dims")
print(f"  Train shape: {X_train_pca.shape}")
print(f"  Val   shape: {X_val_pca.shape}")

# Plot explained variance
plt.figure(figsize=(8, 3))
pca_plot = PCA(n_components=50, random_state=42).fit(train_emb)
plt.plot(np.cumsum(pca_plot.explained_variance_ratio_) * 100, color='steelblue')
plt.axvline(x=N_QUBITS-1, color='coral', linestyle='--', label=f'{N_QUBITS} dims (used)')
plt.axhline(y=80, color='gray', linestyle=':', alpha=0.5, label='80% variance')
plt.xlabel('Number of PCA components')
plt.ylabel('Cumulative variance explained (%)')
plt.title('BERT embedding PCA: cumulative variance')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 5: Normalize to [0, π] for quantum encoding

Quantum rotation gates (RY, RZ) accept angles in [0, 2π].  
Convention: we scale features to [0, π] to keep angles in a clean range.

**Critical**: fit the scaler on training data only, then apply to validation.  
After transforming, clip to [0, π] to handle any out-of-range test values.

In [ ]:
# Fit scaler on train only — prevents data leakage
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)

# Transform val with train statistics
X_val_scaled = scaler.transform(X_val_pca)

# IMPORTANT: clip out-of-range values (test data may slightly exceed [0, pi])
# This happens because MinMaxScaler is fit on train data only
X_val_scaled = np.clip(X_val_scaled, 0, np.pi)

print("After normalization:")
print(f"  Train range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"  Val   range: [{X_val_scaled.min():.4f}, {X_val_scaled.max():.4f}]")
print(f"  (train is exactly [0, pi]; val may differ slightly, clipped to safe range)")

# Final shapes
print(f"\nFinal dataset shapes:")
print(f"  X_train: {X_train_scaled.shape} | y_train: {train_labels.shape}")
print(f"  X_val:   {X_val_scaled.shape}   | y_val:   {val_labels.shape}")
print(f"  Class balance train: {np.mean(train_labels==1):.1%} positive")
print(f"  Class balance val  : {np.mean(val_labels==1):.1%} positive")

---
## Step 6: Classical SVM baselines

We train 3 SVM kernels on the same 8-dim PCA-reduced BERT embeddings.  
These are the benchmarks the quantum kernel must compete with in Week 3.

| Kernel | What it measures |
|--------|-----------------|
| Linear | Dot product — linear separability |
| RBF    | Radial basis: Gaussian similarity, most common baseline |
| Poly   | Polynomial interactions up to degree 3 |

**Note**: we train on 500 samples to match the scale we'll use for quantum experiments.

In [ ]:
# Use a consistent training subset for fair comparison with quantum kernel later
N_TRAIN_SVM = 500  # match this in Week 3 quantum experiments
N_TEST_SVM  = 100

# Subsample if needed
idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), 
                                            min(N_TRAIN_SVM, len(X_train_scaled)), 
                                            replace=False)
idx_te = np.random.RandomState(42).choice(len(X_val_scaled),
                                            min(N_TEST_SVM, len(X_val_scaled)),
                                            replace=False)

X_tr = X_train_scaled[idx_tr]
y_tr = train_labels[idx_tr]
X_te = X_val_scaled[idx_te]
y_te = val_labels[idx_te]

print(f"SVM training on {len(X_tr)} samples, testing on {len(X_te)} samples")
print()

classifiers = {
    'Linear':      SVC(kernel='linear',  C=1.0, random_state=42),
    'RBF':         SVC(kernel='rbf',     C=1.0, gamma='scale', random_state=42),
    'Polynomial':  SVC(kernel='poly',    C=1.0, degree=3, coef0=1, random_state=42),
}

results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'F1':>8} {'Train Time':>12}")
print(f"{'-'*48}")

for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    train_time = time.time() - t0
    y_pred = clf.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    results[name] = {'accuracy': acc, 'f1': f1, 'train_time': train_time, 'clf': clf}
    print(f"{name:<14} {acc:>10.4f} {f1:>8.4f} {train_time:>10.3f}s")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
f1s   = [results[n]['f1'] for n in names]

axes[0].bar(names, accs, color=['steelblue', 'coral', 'mediumseagreen'])
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title('Accuracy by kernel (classical SVM)')
axes[0].set_ylabel('Accuracy')
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=11)

axes[1].bar(names, f1s, color=['steelblue', 'coral', 'mediumseagreen'])
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('F1 score by kernel (classical SVM)')
axes[1].set_ylabel('F1 score')
for i, v in enumerate(f1s):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=11)

plt.suptitle('Classical baselines on 8-dim BERT PCA embeddings (SST-2)')
plt.tight_layout()
plt.show()

---
## Step 7: Effect of PCA dimensions on accuracy

This tells us how much we lose (or gain) by reducing to 4, 8, 12, or 16 dimensions.  
This also shows us which qubit counts to prioritize in Week 4 ablations.

In [ ]:
print("RBF SVM accuracy vs PCA dimensions (train=500, test=100)")
print()
print(f"  {'Dims':>6}  {'Var Explained':>14}  {'Accuracy':>10}  {'F1':>8}")
print(f"  {'-'*46}")

dim_results = {}
for n_dim in pca_dims_to_try:
    pca_d = pca_objects[n_dim]
    X_tr_d = pca_d.transform(train_emb[idx_tr])
    X_te_d = pca_d.transform(val_emb[idx_te])
    sc_d   = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_d = sc_d.fit_transform(X_tr_d)
    X_te_d = np.clip(sc_d.transform(X_te_d), 0, np.pi)
    clf_d  = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
    clf_d.fit(X_tr_d, y_tr)
    y_pred_d = clf_d.predict(X_te_d)
    acc_d = accuracy_score(y_te, y_pred_d)
    f1_d  = f1_score(y_te, y_pred_d)
    var_d = np.sum(pca_objects[n_dim].explained_variance_ratio_) * 100
    dim_results[n_dim] = acc_d
    print(f"  {n_dim:>6}  {var_d:>13.1f}%  {acc_d:>10.4f}  {f1_d:>8.4f}")

---
## Step 8: Save embeddings for Week 3

We save everything needed for the quantum kernel experiments:
- Scaled 8-dim embeddings (ready to feed into quantum circuits)
- Labels
- The fitted PCA and scaler objects (to transform new data consistently)

In [ ]:
import pickle

os.makedirs('qml_project', exist_ok=True)

# Save numpy arrays
np.savez('qml_project/embeddings_week2.npz',
         X_train=X_train_scaled,
         X_val=X_val_scaled,
         y_train=train_labels,
         y_val=val_labels,
         X_tr_subset=X_tr,
         X_te_subset=X_te,
         y_tr_subset=y_tr,
         y_te_subset=y_te)

# Save PCA and scaler for reproducibility
with open('qml_project/pca_scaler.pkl', 'wb') as f:
    pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS}, f)

# Save baseline results for paper table
baseline_summary = {k: {'accuracy': v['accuracy'], 'f1': v['f1']} 
                    for k, v in results.items()}
with open('qml_project/baseline_results.pkl', 'wb') as f:
    pickle.dump(baseline_summary, f)

# Verify reload works
loaded = np.load('qml_project/embeddings_week2.npz')
assert loaded['X_train'].shape == X_train_scaled.shape
assert loaded['y_train'].shape == train_labels.shape
print("Files saved and verified:")
for fname in os.listdir('qml_project'):
    sz = os.path.getsize(f'qml_project/{fname}') / 1024
    print(f"  qml_project/{fname}  ({sz:.0f} KB)")

---
## Week 2 Complete — Summary

### What you built:
- A full data pipeline: SST-2 → BERT embeddings → PCA → normalized features
- Classical SVM baselines (linear, RBF, polynomial) as the comparison benchmark
- All embeddings saved and ready for Week 3

### Key numbers to remember:
- Your classical RBF baseline accuracy is the number to beat (or match) with quantum kernels
- 8-dim PCA captures ~30–35% of BERT embedding variance
- Normalization clips test data to [0, π] safely

### What Week 3 adds:
You will load `qml_project/embeddings_week2.npz`, swap in the quantum kernel function  
from Week 1, compute the kernel matrix on the same 500-sample subset, and compare  
accuracy directly against the classical baselines you just recorded.

### Reading:
- Kübler et al. 2021 — "The inductive bias of quantum kernels": https://arxiv.org/abs/2106.03747
- Schuld 2021 — "Supervised quantum machine learning models are kernel methods": https://arxiv.org/abs/2101.11020